# 03 · Failure and drift (Suchi)
Baseline → logistic → gradient-boosted classifier (planned grid + regularised extension) → calibration and threshold. Then per-band drift.

In [1]:
import json, sys
from pathlib import Path
import pandas as pd
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
R = ROOT / "results"
load = lambda n: json.loads((R / n).read_text())
pd.set_option("display.precision", 4)

In [2]:
cv = load('failure_cv.json'); pd.DataFrame([{'model': r['label'], 'mean PR-AUC': r['mean_pr_auc'], 'mean Brier': r['mean_brier']} for r in cv['logistic'] + cv['tree']])

,model,mean PR-AUC,mean Brier
0,logistic C=0.01,0.7088,0.0894
1,logistic C=0.1,0.7072,0.0890
2,logistic C=1.0,0.7048,0.0892
3,"hgb {'class_weight': None, 'learning_rate': 0....",0.6759,0.0994
4,"hgb {'class_weight': None, 'learning_rate': 0....",0.6618,0.1014
5,"hgb {'class_weight': None, 'learning_rate': 0....",0.6468,0.1043
6,"hgb {'class_weight': None, 'learning_rate': 0....",0.6440,0.1077
7,"hgb {'class_weight': 'balanced', 'learning_rat...",0.6614,0.1157
8,"hgb {'class_weight': 'balanced', 'learning_rat...",0.6721,0.1102
9,"hgb {'class_weight': 'balanced', 'learning_rat...",0.6442,0.1194


In [3]:
x = load('failure_cv_extra.json'); pd.DataFrame([{'model': r['label'], 'mean PR-AUC': r['mean_pr_auc']} for r in x['extra']])

,model,mean PR-AUC
0,"hgb extra {'learning_rate': 0.05, 'max_leaf_no...",0.7105
1,"hgb extra {'learning_rate': 0.03, 'max_leaf_no...",0.7096


## Calibration check on folds (isotonic fitted on the previous fold)

In [4]:
pd.DataFrame([{k: v for k, v in c.items() if not k.startswith('bins')} for c in cv['calibration']])

,fold,brier_raw,brier_iso,ece_raw,ece_iso
0,2,0.0847,0.0859,0.0166,0.0261
1,3,0.1061,0.1035,0.0520,0.0325


In [5]:
pd.Series(load('failure_config.json'))

logit_C                                                                     0.01
tree_params                    {'learning_rate': 0.05, 'max_leaf_nodes': 15, ...
max_iter                                                                     150
keep_tree                                                                  False
tree_beats_logit_every_fold                                                False
logit_mean_pr_auc                                                         0.7088
tree_mean_pr_auc                                                          0.7105
use_isotonic                                                               False
threshold                                                                   0.17
bayes_threshold                                                             0.25
tree_note                      tree_params = best tree over both grids (regul...
dtype: object

## Final test (run once)

In [6]:
F = load('final_metrics.json'); o = F['failure']['overall']
pd.DataFrame({'model': o['model'], 'baseline': o['baseline']})

,model,baseline
n,285306.0000,285306.0000
base_rate,0.3983,0.3983
pr_auc,0.9228,0.5423
roc_auc,0.9385,0.7285
brier,0.0973,0.2470
brier_base,0.2397,0.2397
mean_pred,0.3545,0.2254
ece,0.0438,0.2402
threshold,0.1700,0.1700
precision,0.7686,0.6267


In [7]:
pd.DataFrame(o['ci'], index=['2.5%', '97.5%']).T

,2.5%,97.5%
prauc_model,0.5418,0.9538
prauc_base,0.3958,0.5753
prauc_diff,0.0894,0.4186
brier_diff,-0.2290,-0.0096
base_rate,0.1297,0.5459


In [8]:
pd.DataFrame(F['failure']['calibration_bins']['model'])

,mean_pred,frac_pos,n
0,0.0113,0.0111,28531
1,0.0227,0.0250,28530
2,0.0395,0.0625,28531
3,0.0658,0.0700,28530
4,0.1022,0.1258,28531
5,0.2034,0.3598,28531
6,0.4294,0.5650,28530
7,0.7477,0.8028,28530
8,0.9385,0.9655,28531
9,0.9849,0.9954,28531


## Drift: per band and per day

In [9]:
pd.DataFrame({k: {'PR-AUC model': v['model']['pr_auc'], 'PR-AUC baseline': v['baseline']['pr_auc'], 'base rate': v['model']['base_rate'], 'mean pred': v['model']['mean_pred']} for k, v in F['failure']['slices'].items()}).T

,PR-AUC model,PR-AUC baseline,base rate,mean pred
band_1,0.9242,0.6010,0.4382,0.3255
band_2,0.9812,0.6791,0.5918,0.5542
band_3,0.5823,0.4557,0.1944,0.1922
band_4,0.5074,0.3004,0.1537,0.1612
seen_user,0.9253,0.5472,0.3958,0.3533
unseen_user,0.6514,0.5385,0.5653,0.4393
has_history,0.9229,0.5421,0.3980,0.3542
no_history,0.7670,0.6981,0.6981,0.6535
heavy_user_excluded,0.6137,0.4748,0.1510,0.1384
heavy_user_only,0.9658,0.5303,0.6674,0.5897


In [10]:
d = load('drift.json'); pd.DataFrame(d['failure_by_day'])

,day,n,base_rate,mean_pred_model,mean_pred_baseline,pr_auc_model,pr_auc_baseline
0,56,16442,0.1844,0.1904,0.1828,0.6512,0.5046
1,57,16215,0.1601,0.1643,0.1785,0.7018,0.4830
2,58,17570,0.3405,0.1745,0.1745,0.7478,0.6237
3,59,47298,0.7657,0.5826,0.2040,0.9938,0.9039
4,60,52044,0.7674,0.7611,0.2614,0.9958,0.8020
5,61,21781,0.2705,0.2301,0.2662,0.6769,0.4786
6,62,16875,0.1518,0.1636,0.2170,0.5966,0.3896
7,63,19820,0.2387,0.2507,0.2429,0.7408,0.5499
8,64,19307,0.2013,0.1698,0.2593,0.3963,0.3837
9,65,15794,0.1399,0.1628,0.2274,0.4821,0.3489


In [11]:
pd.Series(d['input_shift'])

role_set_dev_%                     {'tensorflow': 57.92, 'ps+worker': 15.79, 'PyT...
role_set_test_%                    {'tensorflow': 69.42, 'ps+worker': 10.66, 'PyT...
fail_rate_dev                                                                 0.2041
fail_rate_test                                                                0.3983
fail_rate_test_by_band             {'1': 0.4382, '2': 0.5918, '3': 0.1944, '4': 0...
median_logrt_dev                                                              2.8357
median_logrt_test_by_band          {'1': 2.9238, '2': 2.7348, '3': 2.7686, '4': 2...
heavy_user_share_test_by_band_%     {'1': 48.47, '2': 65.74, '3': 29.91, '4': 29.57}
dtype: object